Aunque las gráficas con ACF y PACF son útiles en algunos casos, como por ejemplo cuando hay series que tienen tanto una componente AR como otra MA, pueden resultar difíciles de interpretar.

Por eso en esta prática vamos a tomar un enfoque más cuantitativo.

Hemos visto que los modelos ARIMA (Autoregressive Integrated Moving Average) dependen de 3 hiperparámetros, solemos denotarlo por ARIMA(p,d,q).

Queremos estimar los mejores valores de p, d, q.

Dos modelos ARIMA(p,d,q) y ARIMA(p',d', q') son dos modelos estadísticos que debemos comparar. Para compararlos podemos utilizar dos índices. Cuanto menores sean en general mejor serán los modelos (recordar que esto son estimacione, esto no siempre es cierto, no está de más hacer gráficas y considerar más cosas que mencionaremos más abajo)

1. **AIC** (Akaike Information Criterion)

Se define como

$$
\text{AIC} = -2 \ln(\hat{L}) + 2k
$$

donde

- $\hat{L}$ = valor de la verosimilitud del modelo en los parámetros estimados, es decir lo bien que ajusta.  $-2 \ln(\hat{L})$ disminuye cuando el modelo ajusta mejor

- k = número de parámetros del modelo, $2k$ aumenta cuando el modelo es más complejo

En un ARIMA(p,d,q) se tiene $k=p+q+1$ (AR+MA+ $\sigma^2$)

2. **BIC** (Bayesian Information Criterion)

Similar al anterior:

$$
\text{BIC} = -2 \ln(\hat{L}) + k \ln(n)
$$

La principal diferencia es que aparece $\ln(n)$ penalizando el número de observaciones;
 en nuestro caso al ser dos modelos sobre los mismos datos (mismo n ) lo que hace es penalizar más el valor de k al multiplicarlo por un valor mayor

En general:

AIC: más “permisivo” con parámetros extra → orientado a mejor predicción

BIC: penaliza más los parámetros extra → orientado a modelos más parsimoniosos, es decir más simples.

Estrategia general en esta práctica. Vamos a combinar 3 criterios

para comparar ARIMA(p,d,q) para varios valores p,d,q

1) Vamos a apuntar el valor de AIC  y de BIC (menores --> mejor modelo)

2) Vamos a apuntar el RMSE para un cierto test

3) Como criterio adicional vamos a ver si los residuos son ruido blanco usando el 
test Ljung–Box disponible en `statsmodels.stats.diagnostic import acorr_ljungbox`. 



Estacionariedad

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning # para quitar el warning de kpss
import warnings
warnings.filterwarnings("ignore", category=InterpolationWarning)

def tests_estacionariedad(serie):
    x = np.asarray(serie, float)
    x = x[~np.isnan(x)]
    # --- ADF (H0: raíz unitaria / no estacionaria) ---
    adf_stat, adf_p, adf_lags, adf_n, adf_crit, adf_icbest = adfuller(x, autolag='AIC')
    
    # --- KPSS (H0: estacionaria) ---
    # regression='c'  -> estacionaria alrededor de una constante (nivel)
    # regression='ct' -> estacionaria alrededor de una tendencia
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(x, regression='c', nlags='auto' )
    if kpss_p <= 0.01 and adf_p>0.01: # rechazamos KPSS, pero no ADF
        s = "no estacionario"
    elif kpss_p > 0.01 and adf_p<=0.01: # rechazamos ADF, pero no KPSS 
        s = "estacionario"
    elif kpss_p > 0.01 and adf_p > 0.01: # no se rechaza nada        
        s = "no concluyente"
    else:
        s = "resultados inconsistentes"  # esto no debe suceder...pero sucede
    return s

Código para probar p,(d),q

Devuelve AIC; BIC y test de ruido

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

# quitamos algunos warnings que no afectan pero prueden retrasar el código
from statsmodels.tools.sm_exceptions import ValueWarning,  ConvergenceWarning
warnings.filterwarnings("ignore", category=ValueWarning)
warnings.filterwarnings(
    "ignore",
    message="Non-invertible starting MA parameters found.",
    category=UserWarning
)
warnings.filterwarnings(
    "ignore",
    message="Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.",
    category=UserWarning
)
warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning
)

from tqdm import tqdm


arima = {}

for c in df_train.columns:
    print(c)
    d = 1  # d fijo para esta columna
    resultados = []

    # Serie de trabajo (train solo para esa columna)
    y = df_train[c]

    for p in tqdm(range(1, 4)):      # p = 1..3
        for q in tqdm(range(1, 4)):  # q = 1..3
            # a) Ajuste del modelo ARIMA(p,d,q) sin término de tendencia
            modelo = ARIMA(y, order=(p, d, q), trend="n")
            res = modelo.fit()

            # b) AIC y BIC
            aic = res.aic
            bic = res.bic

            # c) Test Ljung–Box con lags=20 sobre residuos
            # H0: no hay autocorrelación hasta el retardo lags
            lb = acorr_ljungbox(res.resid, lags=[20], return_df=True)
            p_value = lb["lb_pvalue"].iloc[-1]
            estadistico = lb["lb_stat"].iloc[-1]

            resultados.append({
                "p": p,
                "d": d,
                "q": q,
                "AIC": aic,
                "BIC": bic,
                "p_value": p_value,
                "estadistico": estadistico
            })

    # DataFrame de resultados para esta columna
    arima[c] = pd.DataFrame(resultados)


Para encontrar el RMSE

arima_model = ARIMA(y_train, order=order, trend="n")
res = arima_model.fit()

fc = res.get_forecast(steps=1) # predecir mañana

para predecir uno a uno:

In [ ]:
serie_train = train[col]
serie_test  = test[col]

# Logaritmos
serie_train_log = np.log(serie_train)
serie_test_log  = np.log(serie_test)

# 2) Predicción del siguiente

# Aquí guardaremos las predicciones (en log)
pred_log = pd.Series(index=serie_test_log.index, dtype=float) # creamos dataframe con el mismo índice (ahora lleno de NaNs, iremos completando)

# "history" será la serie que se va ampliando (train + test ya utilizado)
history = serie_train_log.copy()

for idx in serie_test_log.index: # se recorre por índice para que los resultados tengan ese mismo índice    
    model = ARIMA(history, order=(1, 0, 0))  
    model_fit = model.fit()

    # Predicción 1 paso adelante
    forecast_log = model_fit.forecast(steps=1)

    # Guardamos la predicción (es una serie de longitud 1)
    pred_log.loc[idx] = forecast_log.iloc[0]

    # Actualizamos "history" con el valor real observado de test
    history.loc[idx] = serie_test_log.loc[idx]

# Volvemos a escala original
pred2 = np.exp(pred_log)


El modelo ARIMA(0,0,0) corresponde a un proceso de ruido blanco con media constante. Su selección como modelo óptimo indica la ausencia de dependencia temporal significativa en la media de los retornos, lo cual es coherente con la hipótesis de eficiencia de mercado. En este contexto, el modelo ARIMA actúa como un modelo base para la media, mientras que la dinámica temporal relevante se encuentra en la varianza, que se analiza mediante modelos GARCH.